# DAPA: create table schemas

Creates every gold table the DAPA dashboards read, **empty**. No data is loaded here.
Each table is a placeholder with its final columns and join keys, so the semantic models and reports
work end to end before any pipeline exists. Ingestion pipelines fill these tables later.

Safe to re-run: a table that already exists is left alone, data included.

| Table | Dashboard area | Filled later by |
|---|---|---|
| `dim_county`, `dim_tract` | Opportunity map | Census geography, ACS 5-year, FDIC household survey, scoring job (F1, F2) |
| `dim_year` | Shared year slicer | Static list |
| `dim_bank`, `dim_branch` | Branch overlay, branch opportunities | FDIC BankFind institutions and branches |
| `fact_branch_deposits`, `fact_branch_events` | Branch opportunities, community evidence | FDIC Summary of Deposits, branch openings and closings |
| `fact_tract_lending` | Community evidence | HMDA, aggregated by tract and year |
| `fact_tract_complaints` | Community evidence | CFPB complaints, ZIP mapped to tract |
| `fact_tract_accounts` | Pilot vs matched control | Bank's own new active accounts per tract (bank tenant only) |
| `dim_event`, `fact_attendee` | Pilot results | Event capture and bank outcomes, bank tenant only (F7, F8) |
| `scenario_ltv` | Scenario view | Azure Machine Learning lifetime value model, labeled assumptions (F11) |
| `agg_playbook` | Mastercard playbook | De-identified aggregate job, groups under 10 attendees suppressed (F10, N2) |

Targeting uses place and need only (N5): no race, ethnicity or national-origin columns exist anywhere.

In [ ]:
SCHEMAS = {
    "dim_county": [
        ("county_fips", "string"), ("county_name", "string"), ("state", "string"),
    ],
    "dim_tract": [
        ("tract_geoid", "string"), ("county_fips", "string"), ("tract_name", "string"),
        ("latitude", "double"), ("longitude", "double"),
        ("households", "bigint"), ("low_income_households", "bigint"), ("limited_english_households", "bigint"),
        ("est_unbanked_households", "double"), ("est_underbanked_households", "double"),
        ("need_score", "double"), ("readiness_score", "double"), ("bank_fit_score", "double"),
        ("opportunity_score", "double"), ("pilot_arm", "string"),
    ],
    "dim_year": [
        ("year", "bigint"),
    ],
    "dim_bank": [
        ("cert", "bigint"), ("fed_rssd", "string"), ("bank_name", "string"), ("assets_000", "double"),
        ("target_issuer", "boolean"), ("is_mdi", "boolean"), ("is_tenant_bank", "boolean"),
    ],
    "dim_branch": [
        ("uninum", "bigint"), ("cert", "bigint"), ("tract_geoid", "string"), ("office_name", "string"),
        ("city", "string"), ("latitude", "double"), ("longitude", "double"), ("servtype_desc", "string"),
    ],
    "fact_branch_deposits": [
        ("uninum", "bigint"), ("year", "bigint"), ("branch_deposits_000", "bigint"),
    ],
    "fact_branch_events": [
        ("uninum", "bigint"), ("tract_geoid", "string"), ("year", "bigint"),
        ("effective_date", "date"), ("event", "string"),
    ],
    "fact_tract_lending": [
        ("tract_geoid", "string"), ("year", "bigint"), ("applications", "bigint"), ("originations", "bigint"),
        ("denials", "bigint"), ("high_cost_originations", "bigint"), ("owner_occupied_units", "bigint"),
    ],
    "fact_tract_complaints": [
        ("tract_geoid", "string"), ("year", "bigint"), ("product_group", "string"), ("complaints", "double"),
    ],
    "fact_tract_accounts": [
        ("tract_geoid", "string"), ("year", "bigint"), ("new_active_accounts", "bigint"),
    ],
    "dim_event": [
        ("event_id", "string"), ("cert", "bigint"), ("tract_geoid", "string"), ("event_date", "date"),
        ("event_type", "string"), ("activities", "string"), ("approach", "string"), ("host_org_type", "string"),
        ("language", "string"), ("product_focus", "string"), ("cost_usd", "double"),
    ],
    "fact_attendee": [
        ("referral_code", "string"), ("event_id", "string"), ("channel", "string"),
        ("consented", "boolean"), ("attended", "boolean"), ("trust_before", "double"), ("trust_after", "double"),
        ("account_opened", "boolean"), ("funded_30d", "boolean"), ("active_90d", "boolean"),
        ("products_held", "double"),
    ],
    "scenario_ltv": [
        ("scenario", "string"), ("year", "bigint"), ("cumulative_ltv_usd", "double"), ("basis", "string"),
    ],
    "agg_playbook": [
        ("area", "string"), ("product_focus", "string"), ("event_type", "string"), ("approach", "string"),
        ("contributor_bank_type", "string"), ("events", "bigint"), ("registrations", "bigint"),
        ("attendees", "bigint"), ("matched_attendees", "bigint"), ("accounts_opened", "bigint"),
        ("active_90d", "bigint"), ("cost_usd", "double"), ("trust_lift_sum", "double"),
        ("survey_responses", "bigint"),
    ],
}

## Create the empty tables

In Fabric this creates Delta tables in the attached lakehouse. Run locally, it only prints the plan.

In [ ]:
for name, columns in SCHEMAS.items():
    ddl = ", ".join(f"`{col}` {dtype}" for col, dtype in columns)
    if "spark" in globals():
        spark.sql(f"CREATE TABLE IF NOT EXISTS {name} ({ddl}) USING DELTA")  # noqa: F821 (Fabric runtime)
        print(f"ready: {name}")
    else:
        print(f"would create {name} ({len(columns)} columns)")